# Movie Recommender System

This project builds a content-based movie recommendation engine using the TMDB 5000 Movies dataset.
It combines NLP-based similarity search with a full ETL pipeline that loads data into SQLite and exports it in multiple formats.

Two recommendation modes are supported:
- **Recommend by title** — given a movie, find the 5 most similar ones
- **Recommend by interest** — given a free-text description, find the best matching movies

## 1. Data Loading

The dataset is loaded from a CSV file containing metadata for 5000 movies from TMDB (The Movie Database).
It includes title, overview, genres, keywords, popularity, vote average and vote count.

In [ ]:
import pandas as pd
import os

file_path = "data/"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    from google.colab import files
    print("Upload dataset")
    files.upload()
    df = pd.read_csv("tmdb_5000_movies.csv")

df.head()

Upload dataset


Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [ ]:
df.shape

(4803, 20)

In [ ]:
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

## 2. Feature Selection and Cleaning

Only four columns are needed for the recommender: title, overview, genres and keywords.
Rows with missing overviews are dropped since the overview is the main source of textual information.

In [ ]:
movies = df[["title", "overview", "genres", "keywords"]]

movies.head()

,title,overview,genres,keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":..."


In [ ]:
movies.isnull().sum()

,0
title,0
overview,3
genres,0
keywords,0


In [ ]:
movies=movies.dropna(subset=["overview"])

movies.shape

(4800, 4)

## 3. Feature Engineering

Genres and keywords are stored as JSON strings in the raw dataset.
I parsed them into Python lists and combined them with the overview into a single "tags" column.
This gives the model a unified text representation of each movie that captures plot, genre and themes.

In [ ]:
import ast

def convert(text):
  L=[]
  for i in ast.literal_eval(text):
    L.append(i["name"])
  return L

movies["genres"]=movies["genres"].apply(convert)
movies["keywords"]=movies["keywords"].apply(convert)

movies.head()

,title,overview,genres,keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."


In [ ]:
movies["tags"]=movies["overview"]+" "+movies["genres"].astype(str)+" "+movies["keywords"].astype(str)
movies[["title","tags"]].head()

,title,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [ ]:
new_df=movies[["title","tags"]]
new_df.head()

,title,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [ ]:
new_df["tags"]=new_df["tags"].apply(lambda x:x.lower())
new_df.head()

/tmp/ipykernel_9003/3068009252.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"]=new_df["tags"].apply(lambda x:x.lower())


,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


## 4. Vectorization

The tags column is converted into numerical vectors using CountVectorizer with a vocabulary of 5000 words.
English stop words (e.g. the, a, is, etc.) are removed to reduce noise.
The result is a matrix of shape (4800, 5000) where each row represents a movie as a bag-of-words vector.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words="english")

vectors = cv.fit_transform(new_df["tags"]).toarray()
vectors.shape

(4800, 5000)

## 5. Cosine Similarity

Cosine similarity measures the angle between two vectors — the closer to 1, the more similar two movies are.
This produces a (4800, 4800) matrix where each cell represents the similarity score between two movies.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
similarity.shape

(4800, 4800)

## 6. Recommendation Functions

Two functions are implemented:
- **recommend()** — takes a movie title and returns the top 5 most similar movies based on cosine similarity
- **recommend_by_interest()** — takes a free-text input, vectorizes it and finds the movies whose tags best match the description

In [ ]:
def recommend(movie):
  movie_index=new_df[new_df["title"]==movie].index[0]
  distances=similarity[movie_index]
  movie_list=sorted(
      list(enumerate(distances)),
      reverse=True,
      key=lambda x: x[1]
      )[1:6]
  for i in movie_list:
    print(new_df.iloc[i[0]].title)
recommend("Avatar")

Aliens
Mission to Mars
Moonraker
Silent Running
Spaceballs


In [ ]:
def recommend_by_interest(user_input):
  user_vector=cv.transform([user_input]).toarray()
  scores=cosine_similarity(user_vector,vectors)
  movie_list=sorted(
      list(enumerate(scores[0])),
      reverse=True,
      key=lambda x: x[1]
  )[0:5]
  for i in movie_list:
    print(new_df.iloc[i[0]].title)

recommend_by_interest("crime mystery detective serial killer")

Suspect Zero
Mindhunters
Se7en
Eye of the Beholder
Switchback


## 7. ETL Pipeline

To go beyond a simple notebook, I built a structured ETL pipeline that loads the movie data into a normalized SQLite database.

The database has three tables:
- **movies** — title, overview, popularity, vote average, vote count
- **genres** — one row per genre per movie (foreign key to movies)
- **keywords** — one row per keyword per movie (foreign key to movies)

This structure allows efficient SQL queries and avoids data redundancy.

In [ ]:
import sqlite3
import logging
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

conn = sqlite3.connect("movies.db")

conn.execute("""
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    overview TEXT,
    popularity REAL,
    vote_average REAL,
    vote_count INTEGER
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS genres (
    movie_id INTEGER,
    genre TEXT,
    FOREIGN KEY(movie_id) REFERENCES movies(id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS keywords (
    movie_id INTEGER,
    keyword TEXT,
    FOREIGN KEY(movie_id) REFERENCES movies(id)
)
""")

conn.commit()
logging.info("Database schema created")

## 8. Extract, Transform, Load

The pipeline is split into three modular functions:
- **extract()** — reads raw data from the DataFrame
- **transform()** — cleans, parses and selects relevant columns
- **load()** — inserts data into the SQLite tables with proper foreign key relationships

Each step logs its progress using Python logging.

In [ ]:
def extract(df):
    logging.info(f"Extracting {len(df)} records from dataframe")
    return df.copy()

def transform(raw_df):
    logging.info("Transforming data...")
    raw_df = raw_df.dropna(subset=["overview"])

    def parse_names(text):
        try:
            return [item["name"] for item in ast.literal_eval(text)]
        except (ValueError, SyntaxError):
            return []

    raw_df = raw_df.copy()
    raw_df["genres_parsed"] = raw_df["genres"].apply(parse_names)
    raw_df["keywords_parsed"] = raw_df["keywords"].apply(parse_names)

    cols = ["title", "overview", "popularity", "vote_average", "vote_count",
            "genres_parsed", "keywords_parsed"]
    available = [c for c in cols if c in raw_df.columns]
    clean = raw_df[available].reset_index(drop=True)

    logging.info(f"Transformation complete: {len(clean)} records")
    return clean

def load(clean_df, conn):
    logging.info("Loading data into SQLite...")
    conn.execute("DELETE FROM genres")
    conn.execute("DELETE FROM keywords")
    conn.execute("DELETE FROM movies")
    conn.commit()

    for _, row in clean_df.iterrows():
        cursor = conn.execute(
            "INSERT INTO movies (title, overview, popularity, vote_average, vote_count) VALUES (?, ?, ?, ?, ?)",
            (
                row["title"],
                row.get("overview", ""),
                row.get("popularity", None),
                row.get("vote_average", None),
                row.get("vote_count", None),
            )
        )
        movie_id = cursor.lastrowid

        for g in row.get("genres_parsed", []):
            conn.execute("INSERT INTO genres (movie_id, genre) VALUES (?, ?)", (movie_id, g))

        for k in row.get("keywords_parsed", []):
            conn.execute("INSERT INTO keywords (movie_id, keyword) VALUES (?, ?)", (movie_id, k))

    conn.commit()
    logging.info(f"Loaded {len(clean_df)} movies into database")

raw = extract(df)
clean = transform(raw)
load(clean, conn)

## 9. SQL Queries

With the data loaded into SQLite, I can run analytical queries directly on the database.
These queries demonstrate standard SQL operations: GROUP BY, ORDER BY, WHERE filters, JOIN and HAVING.

In [ ]:
top_genres = pd.read_sql("""
SELECT genre, COUNT(*) as count
FROM genres
GROUP BY genre
ORDER BY count DESC
LIMIT 10
""", conn)

print(top_genres.to_string(index=False))

          genre  count
          Drama   2296
         Comedy   1722
       Thriller   1274
         Action   1154
        Romance    894
      Adventure    790
          Crime    696
Science Fiction    535
         Horror    519
         Family    513


In [ ]:
top_movies = pd.read_sql("""
SELECT m.title, m.vote_average, m.vote_count
FROM movies m
WHERE m.vote_count > 500
ORDER BY m.vote_average DESC
LIMIT 10
""", conn)

print(top_movies.to_string(index=False))

                   title  vote_average  vote_count
The Shawshank Redemption           8.5        8205
           The Godfather           8.4        5893
              Fight Club           8.3        9413
        Schindler's List           8.3        4329
           Spirited Away           8.3        3840
  The Godfather: Part II           8.3        3338
            Pulp Fiction           8.3        8428
                Whiplash           8.3        4254
         The Dark Knight           8.2       12002
          The Green Mile           8.2        4048


In [ ]:
genre_stats = pd.read_sql("""
SELECT g.genre, AVG(m.vote_average) as avg_rating, COUNT(*) as total_movies
FROM genres g
JOIN movies m ON g.movie_id = m.id
GROUP BY g.genre
HAVING total_movies > 10
ORDER BY avg_rating DESC
""", conn)

print(genre_stats.to_string(index=False))

          genre  avg_rating  total_movies
        History    6.719797           197
            War    6.713889           144
          Drama    6.388197          2296
          Music    6.355676           185
        Foreign    6.352941            34
      Animation    6.341453           234
    Documentary    6.285185           108
          Crime    6.274138           696
        Romance    6.207718           894
        Mystery    6.183908           348
        Western    6.178049            82
      Adventure    6.156962           790
        Fantasy    6.096698           424
         Family    6.029630           513
       Thriller    6.010989          1274
Science Fiction    6.005607           535
         Action    5.989515          1154
         Comedy    5.945587          1722
         Horror    5.626590           519


## 10. Export to Multiple Formats

The cleaned dataset is exported in three formats:
- **CSV** — universal format, readable by any tool
- **JSON** — standard format for APIs and web applications
- **Parquet** — columnar format optimized for big data tools (Spark, BigQuery, AWS S3)

Parquet is significantly smaller and faster to read than CSV for large datasets.

In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

os.makedirs("output", exist_ok=True)

export_df = pd.read_sql("SELECT title, overview, popularity, vote_average, vote_count FROM movies", conn)

export_df.to_csv("output/movies_clean.csv", index=False)
logging.info("Exported CSV")

export_df.to_json("output/movies_clean.json", orient="records", indent=2)
logging.info("Exported JSON")

table = pa.Table.from_pandas(export_df)
pq.write_table(table, "output/movies_clean.parquet")
logging.info("Exported Parquet")

print(f"CSV size:     {os.path.getsize('output/movies_clean.csv') / 1024:.1f} KB")
print(f"JSON size:    {os.path.getsize('output/movies_clean.json') / 1024:.1f} KB")
print(f"Parquet size: {os.path.getsize('output/movies_clean.parquet') / 1024:.1f} KB")

CSV size:     1604.3 KB
JSON size:    2053.7 KB
Parquet size: 1132.3 KB


In [ ]:
parquet_back = pd.read_parquet("output/movies_clean.parquet", engine="pyarrow")
print(parquet_back.shape)
print(parquet_back.head())

(4800, 5)
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  popularity  \
0  In the 22nd century, a paraplegic Marine is di...  150.437577   
1  Captain Barbossa, long believed to be dead, ha...  139.082615   
2  A cryptic message from Bond’s past sends him o...  107.376788   
3  Following the death of District Attorney Harve...  112.312950   
4  John Carter is a war-weary, former military ca...   43.926995   

   vote_average  vote_count  
0           7.2       11800  
1           6.9        4500  
2           6.3        4466  
3           7.6        9106  
4           6.1        2124  


## 11. Save Model Files

The vectorizer, similarity matrix and movie data are saved as pickle files.
These are loaded by the Streamlit app at startup, so the model does not need to be retrained each time.

In [ ]:
import pickle

pickle.dump(new_df.to_dict(), open('movies_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(cv, open('vectorizer.pkl', 'wb'))
pickle.dump(vectors, open('vectors.pkl', 'wb'))

print('Saved: movies_dict.pkl, similarity.pkl, vectorizer.pkl, vectors.pkl')
print('Run: streamlit run app.py')

Saved: movies_dict.pkl, similarity.pkl, vectorizer.pkl, vectors.pkl
Run: streamlit run app.py


## 12. Streamlit App

The cell below writes the Streamlit app to disk as app.py.
To launch it locally run: streamlit run app.py
To launch it in Google Colab run the two cells below (pyngrok setup required).

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title="Movie Recommender", layout="centered")

@st.cache_data
def load_data():
    movies = pd.DataFrame(pickle.load(open("movies_dict.pkl", "rb")))
    similarity = pickle.load(open("similarity.pkl", "rb"))
    cv = pickle.load(open("vectorizer.pkl", "rb"))
    vectors = pickle.load(open("vectors.pkl", "rb"))
    return movies, similarity, cv, vectors

movies, similarity, cv, vectors = load_data()

st.title("Movie Recommender System")

tab1, tab2 = st.tabs(["Recommend by title", "Recommend by interest"])

with tab1:
    movie = st.selectbox("Select a movie", sorted(movies["title"].values))
    n = st.slider("Number of recommendations", 3, 10, 5)
    if st.button("Recommend", key="btn1"):
        idx = movies[movies["title"] == movie].index[0]
        distances = similarity[idx]
        results = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:n+1]
        for rank, (i, score) in enumerate(results, 1):
            st.write(f"**{rank}. {movies.iloc[i].title}**")
            st.progress(float(score))

with tab2:
    interest = st.text_input("Describe what you want to watch", "crime mystery detective")
    n2 = st.slider("Number of recommendations", 3, 10, 5, key="s2")
    if st.button("Search", key="btn2"):
        user_vec = cv.transform([interest.lower()]).toarray()
        scores = cosine_similarity(user_vec, vectors)[0]
        results = sorted(list(enumerate(scores)), reverse=True, key=lambda x: x[1])[:n2]
        for rank, (i, score) in enumerate(results, 1):
            st.write(f"**{rank}. {movies.iloc[i].title}**")
            st.progress(float(min(score, 1.0)))

Writing app.py


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
import subprocess
import time

ngrok.set_auth_token("NGROK_TOKEN")

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(5)

public_url = ngrok.connect(8501)
print(f"Live app: {public_url}")